In [16]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd
import yfinance as yf

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import DB_PATH

In [24]:
TICKERS = ["SPY", "QQQ", "GLD", "TLT", "EURUSD=X", "^VIX"]
START_DATE = "2015-01-01"
INTERVAL = "1d"
TABLE_NAME = "daily_prices"

In [19]:
data = yf.download(
    TICKERS,
    start=START_DATE,
    interval=INTERVAL,
    auto_adjust=True,
    group_by="ticker",
)

[*********************100%***********************]  6 of 6 completed


In [36]:
# change multi-index yf data into single index table (reshape / normalize
# Reshape yfinance MultiIndex data into SQL-friendly long format.

records = (
    data
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(columns={
        "Date": "date",
        "level_1": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
)

records.head()

Price,date,Ticker,open,high,low,close,volume
0,2015-01-01,EURUSD=X,1.209863,1.209863,1.209863,1.209863,0.0
1,2015-01-01,QQQ,NaN,NaN,NaN,NaN,NaN
2,2015-01-01,GLD,NaN,NaN,NaN,NaN,NaN
3,2015-01-01,^VIX,NaN,NaN,NaN,NaN,NaN
4,2015-01-01,TLT,NaN,NaN,NaN,NaN,NaN


In [25]:
with sqlite3.connect(DB_PATH) as conn:
    records.to_sql(
        TABLE_NAME,
        conn,
        if_exists="replace",
        index=False,
    )

In [27]:
with sqlite3.connect(DB_PATH) as conn:
    check = pd.read_sql(
        f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY date, ticker
        LIMIT 10
        """,
        conn,
    )
check

,date,Ticker,open,high,low,close,volume
0,2015-01-01 00:00:00,EURUSD=X,1.209863,1.209863,1.209863,1.209863,0.0
1,2015-01-01 00:00:00,GLD,NaN,NaN,NaN,NaN,NaN
2,2015-01-01 00:00:00,QQQ,NaN,NaN,NaN,NaN,NaN
3,2015-01-01 00:00:00,SPY,NaN,NaN,NaN,NaN,NaN
4,2015-01-01 00:00:00,TLT,NaN,NaN,NaN,NaN,NaN
5,2015-01-01 00:00:00,^VIX,NaN,NaN,NaN,NaN,NaN
6,2015-01-02 00:00:00,EURUSD=X,1.208868,1.208956,1.201080,1.208941,0.0
7,2015-01-02 00:00:00,GLD,112.489998,114.800003,112.320000,114.080002,7109600.0
8,2015-01-02 00:00:00,QQQ,95.419153,95.823778,94.205263,94.665070,31314600.0
9,2015-01-02 00:00:00,SPY,170.911790,171.325861,169.089869,170.125046,121465900.0


In [28]:
with sqlite3.connect(DB_PATH) as conn:
    summary = pd.read_sql(
        f"""
        SELECT
            ticker,
            COUNT(*) AS rows,
            MIN(date) AS start_date,
            MAX(date) AS end_date
        FROM {TABLE_NAME}
        GROUP BY ticker
        ORDER BY ticker
        """,
        conn,
    )

summary

,Ticker,rows,start_date,end_date
0,EURUSD=X,2981,2015-01-01 00:00:00,2026-06-10 00:00:00
1,GLD,2981,2015-01-01 00:00:00,2026-06-10 00:00:00
2,QQQ,2981,2015-01-01 00:00:00,2026-06-10 00:00:00
3,SPY,2981,2015-01-01 00:00:00,2026-06-10 00:00:00
4,TLT,2981,2015-01-01 00:00:00,2026-06-10 00:00:00
5,^VIX,2981,2015-01-01 00:00:00,2026-06-10 00:00:00


In [32]:
records.isna().sum() # default axis=0 ie columns

Price
date        0
Ticker      0
open      537
high      537
low       537
close     537
volume    533
dtype: int64

In [33]:
records[records.isna().any(axis=1)].head(20)

Price,date,Ticker,open,high,low,close,volume
1,2015-01-01,QQQ,NaN,NaN,NaN,NaN,NaN
2,2015-01-01,GLD,NaN,NaN,NaN,NaN,NaN
3,2015-01-01,^VIX,NaN,NaN,NaN,NaN,NaN
4,2015-01-01,TLT,NaN,NaN,NaN,NaN,NaN
5,2015-01-01,SPY,NaN,NaN,NaN,NaN,NaN
73,2015-01-19,QQQ,NaN,NaN,NaN,NaN,NaN
74,2015-01-19,GLD,NaN,NaN,NaN,NaN,NaN
75,2015-01-19,^VIX,NaN,NaN,NaN,NaN,NaN
76,2015-01-19,TLT,NaN,NaN,NaN,NaN,NaN
77,2015-01-19,SPY,NaN,NaN,NaN,NaN,NaN


In [34]:
records[records["volume"].notna() & records["open"].isna()]

Price,date,Ticker,open,high,low,close,volume
17875,2026-06-09,QQQ,NaN,NaN,NaN,NaN,87562233.0
17876,2026-06-09,GLD,NaN,NaN,NaN,NaN,9484805.0
17878,2026-06-09,TLT,NaN,NaN,NaN,NaN,21665239.0
17879,2026-06-09,SPY,NaN,NaN,NaN,NaN,86844734.0


In [35]:
records[records["date"] == records["date"].max()]

Price,date,Ticker,open,high,low,close,volume
17880,2026-06-10,EURUSD=X,1.154201,1.156203,1.153669,1.155802,0.0
17881,2026-06-10,QQQ,NaN,NaN,NaN,NaN,NaN
17882,2026-06-10,GLD,NaN,NaN,NaN,NaN,NaN
17883,2026-06-10,^VIX,20.100000,20.100000,20.059999,20.059999,0.0
17884,2026-06-10,TLT,NaN,NaN,NaN,NaN,NaN
17885,2026-06-10,SPY,NaN,NaN,NaN,NaN,NaN
